# Circular Section Design Checks (Orr 2012)

This notebook demonstrates `CircularSectionCheck` — an EC2-compliant wrapper for
bending, shear, and cracking checks on circular RC sections (piles, columns, etc.).

**Key features:**
- Shear reinforcement efficiency factors **λ₁** (closed links) and **λ₂** (spirals)
- **Equivalent web width** for concrete strut crushing (V_Rd,max)
- **Uncracked V_Rd,c** from principal stress approach (Eq.17)
- Lever arm **z** from first-principles (capped to 0.9d as upper limit)
- Optional **k_f** factor for cast-in-place piles (EC2 §2.4.2.5(2))
- **Tension shift** using circular geometry for cot(θ)

**Reference:**  
Orr, J.J. (2012). *Shear design of circular concrete sections using the Eurocode 2 truss model.* University of Bath.

In [33]:
import warnings
import numpy as np

# Section geometry
from materials.reinforced_concrete.geometry import (
    create_circular_section,
    create_circular_perimeter_rebars,
)

# Materials
from materials.reinforced_concrete.materials import (
    ConcreteMaterial,
    Rebar,
    ShearRebar,
)

# Code checks
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    CircularSectionCheck,
    ShearLoadCase,
)

---
## 1. Create a Circular RC Section

We'll model a typical 600mm diameter pile with:
- 10H20 bars evenly spaced around the perimeter
- H12@200 closed shear links
- C30/37 concrete
- 50mm cover to link outer face

**Important:** `create_circular_section()` uses `hook_ref=1` by default (origin at bottom-left),
so the section centre is at `(pile_dia/2, D/2)`. `create_circular_perimeter_rebars()` must be called
with `origin=(D/2, D/2)` to match.

In [34]:
# Section parameters
pile_dia = 600       # Diameter (mm)
cover = 50    # Cover to link outer face (mm)
link_dia = 16
bar_dia = 20
n_bars = 10

# Create section (origin at bottom-left, centre at D/2, D/2)
section = create_circular_section(
    diameter=pile_dia,
    section_name=f"{pile_dia}mm Pile",
)

# Add perimeter reinforcement (origin must match section centre)
rebar = Rebar(diameter=bar_dia)
rebar_cover = cover + link_dia
rebar_group = create_circular_perimeter_rebars(
    rebar=rebar,
    diameter=pile_dia,
    cover=rebar_cover,
    n_bars=n_bars,
    origin=(pile_dia / 2, pile_dia / 2),
)
section.add_rebar_group(rebar_group)

# Materials
concrete = ConcreteMaterial(grade="C30/37")
links = ShearRebar(diameter=link_dia, link_spacing=200, n_legs=2, grade="B500B")

# Display
section.plot(concrete=concrete)

print(f"Section: {section.section_name}")
print(f"Diameter: {pile_dia} mm")
print(f"Concrete: {concrete.grade} (f_ck = {concrete.f_ck} MPa, f_cd = {concrete.f_cd:.2f} MPa)")
print(f"Longitudinal steel: {n_bars}H{bar_dia} (A_s = {n_bars * rebar.area:.0f} mm\u00b2)")
print(f"Shear links: H{link_dia}@{links.link_spacing} ({links.n_legs}-leg closed links)")
print(f"Reinforcement ratio: {section.reinforcement_ratio:.3%}")

Section: 600mm Pile
Diameter: 600 mm
Concrete: C30/37 (f_ck = 30.0 MPa, f_cd = 20.00 MPa)
Longitudinal steel: 10H20 (A_s = 3142 mm²)
Shear links: H16@200.0 (2-leg closed links)
Reinforcement ratio: 1.113%


---
## 2. Instantiate CircularSectionCheck

The check wraps `BendingCheck`, `ShearCheck`, and `CrackingCheck` with circular-specific
modifications. Key defaults:
- `cap_lever_arm=True` (caps upper limit of lever_arm to 0.9d)
- `use_rigorous=True` (lever arm from force centroids, not simplified)
- SLS cracking uses original concrete uses alpha_cc but not gamma_c so need for no need k_f

In [35]:
check = CircularSectionCheck(
    section=section,
    concrete=concrete,
    diameter=pile_dia,
    cover=cover,
    shear_reinforcement=links,
)

print(f"r_sv (auto): {check.r_sv:.1f} mm")
print(f"  = D/2 - cover - link_dia/2 = {pile_dia/2} - {cover} - {link_dia/2} = {pile_dia/2 - cover - link_dia/2:.1f} mm")
print(f"\u03bb\u2082 (closed links): {check.calculate_lambda_2():.3f}")
print(f"k_f applied: {check.apply_k_f}")

r_sv (auto): 242.0 mm
  = D/2 - cover - link_dia/2 = 300.0 - 50 - 8.0 = 242.0 mm
λ₂ (closed links): 1.000
k_f applied: False


---
## 3. Bending Check

The bending check forwards to `BendingCheck` with `iterate_z=True` by default.
The sub-check is accessible via `check.bending` for M-N plots and capacity queries.

### 3.1 M-N Interaction Diagram

In [36]:
# Plot the M-N diagram via the sub-check
check.bending.plot_mn(
    title=f"M-N Interaction Diagram - {pile_dia}mm Circular Pile",
    show_metadata=True,
    show=False,
)

### 3.2 Check Multiple Load Cases

In [37]:
load_cases = [
    {"name": "LC1: Pure bending",       "M_Ed": 200, "N_Ed": 0},
    {"name": "LC2: Axial + bending",    "M_Ed": 150, "N_Ed": 800},
    {"name": "LC3: High compression",   "M_Ed": 100, "N_Ed": 2000},
    {"name": "LC4: Bending + tension",  "M_Ed": 120, "N_Ed": -200},
]

print("Bending Check Results")
print("=" * 70)

for lc in load_cases:
    result = check.perform_bending_check(M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
    d = result.details
    print(f"\n{lc['name']}:")
    print(f"  M_Ed = {lc['M_Ed']:6.1f} kN\u00b7m, N_Ed = {lc['N_Ed']:6.1f} kN")
    print(f"  M_Rd = {d['M_Rd']:6.1f} kN\u00b7m, N_Rd = {d['N_Rd']:6.1f} kN")
    print(f"  Utilization: {result.utilization:.1%} [{result.status.value.upper()}]")

Bending Check Results

LC1: Pure bending:
  M_Ed =  200.0 kN·m, N_Ed =    0.0 kN
  M_Rd =  290.5 kN·m, N_Rd =    0.0 kN
  Utilization: 68.8% [PASS]

LC2: Axial + bending:
  M_Ed =  150.0 kN·m, N_Ed =  800.0 kN
  M_Rd =  491.5 kN·m, N_Rd = 2621.3 kN
  Utilization: 30.5% [PASS]

LC3: High compression:
  M_Ed =  100.0 kN·m, N_Ed = 2000.0 kN
  M_Rd =  270.3 kN·m, N_Rd = 5406.0 kN
  Utilization: 37.0% [PASS]

LC4: Bending + tension:
  M_Ed =  120.0 kN·m, N_Ed = -200.0 kN
  M_Rd =  225.5 kN·m, N_Rd = -375.8 kN
  Utilization: 53.2% [PASS]


### 3.3 Bending with Tension Shift

When shear is present, the tension shift rule (EC2 §9.2.1.3) increases the
effective moment. For circular sections, the standard `calculate_section_breadth()`
gives the wrong web width. `CircularSectionCheck` automatically computes
cot(θ) from the circular equivalent web width and passes it as `cot_theta_override`,
bypassing the internal rectangular b_w entirely.

In [38]:
# Without tension shift (no V_Ed / M_cap)
result_no_shift = check.perform_bending_check(M_Ed=200, N_Ed=500)

# With tension shift (V_Ed + M_cap triggers circular cot_theta auto-computation)
M_cap = check.bending.get_moment_capacity(N_Ed=500)[0]  # positive capacity
result_with_shift = check.perform_bending_check(
    M_Ed=200, N_Ed=500, V_Ed=300, M_cap=M_cap,
)

print("Tension Shift Comparison")
print("=" * 58)
print(f"M_Ed = 200 kN\u00b7m, N_Ed = 500 kN, V_Ed = 300 kN")
print(f"M_cap = {M_cap:.1f} kN\u00b7m")
print()
print(f"{'Parameter':<30} | {'No shift':>10} | {'With shift':>10}")
print("-" * 58)
print(f"{'Utilization':<30} |  {result_no_shift.utilization:>9.1%} |  {result_with_shift.utilization:>9.1%}")
print(f"{'Status':<30} | {result_no_shift.status.value:>10} | {result_with_shift.status.value:>10}")

Tension Shift Comparison
M_Ed = 200 kN·m, N_Ed = 500 kN, V_Ed = 300 kN
M_cap = 370.4 kN·m

Parameter                      |   No shift | With shift
----------------------------------------------------------
Utilization                    |      46.1% |      90.0%
Status                         |       pass |       pass


---
## 4. Shear Check (Orr 2012)

The shear check is a custom implementation using:
- **λ₁** (link efficiency) from numerical integration (Eq.6) or simplified 0.85
- **λ₂** = 1.0 for closed links (< 1.0 for spirals)
- **Circular b_w** = min(b_wc, b_wt) from Eq.10–13
- **V_Rd,c uncracked** from principal stress (Eq.17) when section is uncracked

### 4.1 Basic Shear Check

In [39]:
shear_lc = ShearLoadCase(V_Ed=250, M_Ed=50, N_Ed=0)

result = check.perform_shear_check(load_case=shear_lc)
d = result.details

print("Shear Check Result")
print("=" * 55)
print(f"V_Ed = {shear_lc.V_Ed} kN, M_Ed = {shear_lc.M_Ed} kN\u00b7m, N_Ed = {shear_lc.N_Ed} kN")
print()
print(f"Effective depth d:    {d['d']:.1f} mm")
print(f"Lever arm z:          {d['z']:.1f} mm")
print(f"\u03c3_cp:                 {d['sigma_cp']:.2f} MPa")
print()
if 'lambda_1' in d:
    print(f"\u03bb\u2081 (link efficiency):  {d['lambda_1']:.4f}")
    print(f"\u03bb\u2082 (spiral factor):   {d['lambda_2']:.4f}")
    print(f"b_w (circular):       {d['b_w']:.1f} mm (b_wc={d['b_wc']:.1f}, b_wt={d['b_wt']:.1f})")
    print(f"cot(\u03b8):               {d['cot_theta']:.3f}")
    print(f"\u03b1_cw:                 {d['alpha_cw']:.3f}")
    print(f"\u03bd\u2081:                   {d['nu_1']:.3f}")
    print(f"V_Rd,c (selected):   {d['V_Rd_c']:.1f} kN")
    print(f"V_Rd,c,cracked:      {d['V_Rd_c_cracked']:.1f} kN")
    print(f"V_Rd,c,uncracked:    {d['V_Rd_c_uncracked']:.1f} kN")
    print(f"V_Rd,s:               {d['V_Rd_s']:.1f} kN")
    print(f"V_Rd,max:             {d['V_Rd_max']:.1f} kN")
print()
print(f"Capacity:             {result.capacity:.1f} kN")
print(f"Utilization:          {result.utilization:.1%} [{result.status.value.upper()}]")

Shear Check Result
V_Ed = 250.0 kN, M_Ed = 50.0 kN·m, N_Ed = 0.0 kN

Effective depth d:    414.9 mm
Lever arm z:          372.9 mm
σ_cp:                 0.00 MPa

λ₁ (link efficiency):  0.8057
λ₂ (spiral factor):   1.0000
b_w (circular):       306.1 mm (b_wc=306.1, b_wt=426.0)
cot(θ):               2.500
α_cw:                 1.000
ν₁:                   0.528
V_Rd,c (selected):   100.7 kN
V_Rd,c,cracked:      100.7 kN
V_Rd,c,uncracked:    286.6 kN
V_Rd,s:               656.6 kN
V_Rd,max:             415.6 kN

Capacity:             415.6 kN
Utilization:          60.1% [PASS]


### 4.2 V_Rd,c Reporting Policy

Circular shear now uses reinforced capacity as governing in all cases:
`V_Rd = min(V_Rd,s, V_Rd,max)`.

Both `V_Rd,c,cracked` and `V_Rd,c,uncracked` are still reported. The
`use_uncracked_V_Rd_c` flag only changes which `V_Rd,c` value is selected for
reporting and spacing-rule input.

In [40]:
# Compare selected V_Rd,c policy for the same load case
lc_low_M = ShearLoadCase(V_Ed=200, M_Ed=30, N_Ed=1000)

# Default policy: cracked V_Rd,c selected
result_default = check.perform_shear_check(load_case=lc_low_M)

result_uncracked = check.perform_shear_check(load_case=lc_low_M, use_uncracked_V_Rd_c=True)

print("V_Rd,c Policy Comparison")
print("=" * 80)
print(f"V_Ed = {lc_low_M.V_Ed} kN, M_Ed = {lc_low_M.M_Ed} kN\u00b7m, N_Ed = {lc_low_M.N_Ed} kN")
print()
print(f"{'Case':<24} | {'V_Rd,c sel':>12} | {'V_Rd,s':>10} | {'V_Rd,max':>10} | {'Capacity':>10}")
print("-" * 80)
print(f"{'default (cracked)':<24} | {result_default.details['V_Rd_c']:>12.1f} | {result_default.details['V_Rd_s']:>10.1f} | {result_default.details['V_Rd_max']:>10.1f} | {result_default.capacity:>10.1f}")
print(f"{'use_uncracked_V_Rd_c':<24} | {result_uncracked.details['V_Rd_c']:>12.1f} | {result_uncracked.details['V_Rd_s']:>10.1f} | {result_uncracked.details['V_Rd_max']:>10.1f} | {result_uncracked.capacity:>10.1f}")


V_Rd,c Policy Comparison
V_Ed = 200.0 kN, M_Ed = 30.0 kN·m, N_Ed = 1000.0 kN

Case                     |   V_Rd,c sel |     V_Rd,s |   V_Rd,max |   Capacity
--------------------------------------------------------------------------------
default (cracked)        |        120.5 |      656.7 |      414.0 |      414.0
use_uncracked_V_Rd_c     |        545.5 |      656.7 |      414.0 |      414.0


C:\Users\user\Repo\Scripts\section_design_checks\materials\reinforced_concrete\code_checks\ec2_2004\shear_check.py:506: UserWarning:

Lever arm fallback to 0.9d: unable to compute a meaningful tension/compression centroid lever arm for this strain state.



### 4.3 Multiple Shear Load Cases

In [41]:
shear_cases = [
    {"name": "Low shear, high axial",   "V_Ed": 100, "M_Ed": 50,  "N_Ed": 1500},
    {"name": "Moderate shear",           "V_Ed": 250, "M_Ed": 150, "N_Ed": 500},
    {"name": "High shear",               "V_Ed": 400, "M_Ed": 200, "N_Ed": 300},
    {"name": "Shear with tension",       "V_Ed": 150, "M_Ed": 100, "N_Ed": -100},
]

print("Shear Check Results - Multiple Load Cases")
print("=" * 90)

for sc in shear_cases:
    lc = ShearLoadCase(V_Ed=sc["V_Ed"], M_Ed=sc["M_Ed"], N_Ed=sc["N_Ed"])
    result = check.perform_shear_check(load_case=lc)
    d = result.details

    print(f"\n{sc['name']}:")
    print(f"  V_Ed={sc['V_Ed']:3.0f} kN, M_Ed={sc['M_Ed']:3.0f} kN\u00b7m, N_Ed={sc['N_Ed']:5.0f} kN")
    if 'lambda_1' in d:
        print(f"  \u03bb\u2081={d['lambda_1']:.3f}, b_w={d['b_w']:.0f} mm, cot(\u03b8)={d['cot_theta']:.2f}")
        print(f"  V_Rd,s={d['V_Rd_s']:.1f} kN, V_Rd,max={d['V_Rd_max']:.1f} kN")
    elif d.get('V_Rd_c') is not None:
        print(f"  V_Rd,c (uncracked) = {d['V_Rd_c']:.1f} kN")
    print(f"  Utilization: {result.utilization:.1%} [{result.status.value.upper()}]")

Shear Check Results - Multiple Load Cases

Low shear, high axial:
  V_Ed=100 kN, M_Ed= 50 kN·m, N_Ed= 1500 kN
  λ₁=0.805, b_w=304 mm, cot(θ)=2.50
  V_Rd,s=656.7 kN, V_Rd,max=414.0 kN
  Utilization: 24.2% [PASS]

Moderate shear:
  V_Ed=250 kN, M_Ed=150 kN·m, N_Ed=  500 kN
  λ₁=0.848, b_w=363 mm, cot(θ)=2.50
  V_Rd,s=655.9 kN, V_Rd,max=467.7 kN
  Utilization: 53.5% [PASS]

High shear:
  V_Ed=400 kN, M_Ed=200 kN·m, N_Ed=  300 kN
  λ₁=0.834, b_w=345 mm, cot(θ)=2.50
  V_Rd,s=656.7 kN, V_Rd,max=452.7 kN
  Utilization: 88.4% [PASS]

Shear with tension:
  V_Ed=150 kN, M_Ed=100 kN·m, N_Ed= -100 kN
  λ₁=0.810, b_w=312 mm, cot(θ)=2.50
  V_Rd,s=656.8 kN, V_Rd,max=421.7 kN
  Utilization: 35.6% [PASS]


### 4.4 λ₁ Efficiency Factor — Numerical vs Simplified

The link efficiency factor λ₁ accounts for the fact that only a component of the
force in a circular link resists vertical shear. The numerical integration (Eq.6)
is more accurate than the simplified value of 0.85.

In [42]:
# Create a simplified check for comparison
check_simplified = CircularSectionCheck(
    section=section,
    concrete=concrete,
    diameter=pile_dia,
    cover=cover,
    shear_reinforcement=links,
    use_simplified_lambda_1=True,   # Use 0.85 instead of Eq.6
)

test_lc = ShearLoadCase(V_Ed=250, M_Ed=150, N_Ed=500)

result_numerical = check.perform_shear_check(load_case=test_lc)
result_simplified = check_simplified.perform_shear_check(load_case=test_lc)

print("\u03bb\u2081 Comparison: Numerical (Eq.6) vs Simplified (0.85)")
print("=" * 55)
print(f"{'Parameter':<25} | {'Numerical':>12} | {'Simplified':>12}")
print("-" * 55)
print(f"{'\u03bb\u2081':<25} | {result_numerical.details['lambda_1']:>12.4f} | {result_simplified.details['lambda_1']:>12.4f}")
print(f"{'V_Rd,s (kN)':<25} | {result_numerical.details['V_Rd_s']:>12.1f} | {result_simplified.details['V_Rd_s']:>12.1f}")
print(f"{'V_Rd,max (kN)':<25} | {result_numerical.details['V_Rd_max']:>12.1f} | {result_simplified.details['V_Rd_max']:>12.1f}")
print(f"{'Utilization':<25} | {result_numerical.utilization:>11.1%} | {result_simplified.utilization:>11.1%}")

λ₁ Comparison: Numerical (Eq.6) vs Simplified (0.85)
Parameter                 |    Numerical |   Simplified
-------------------------------------------------------
λ₁                        |       0.8484 |       0.8500
V_Rd,s (kN)               |        655.9 |        657.1
V_Rd,max (kN)             |        467.7 |        467.7
Utilization               |       53.5% |       53.5%


---
## 5. Cracking Check

The cracking check is a pass-through wrapper to `CrackingCheck` (no circular
modifications needed). The sub-check is accessible via `check.cracking`.

### 5.1 Basic Cracking Check

In [43]:
# SLS load cases
sls_cases = [
    {"name": "Low SLS moment",       "M_Ed": 40,  "N_Ed": 0},
    {"name": "Service moment",        "M_Ed": 80,  "N_Ed": 0},
    {"name": "With axial compression","M_Ed": 100,  "N_Ed": 200},
    {"name": "High moment + axial",   "M_Ed": 120, "N_Ed": 200},
]

M_cr = check.cracking.find_cracking_moment()
print(f"Cracking moment M_cr (N=0): {M_cr:.1f} kN\u00b7m")
print()

print("Cracking Check Results")
print("=" * 70)

for sc in sls_cases:
    result = check.perform_cracking_check(M_Ed=sc["M_Ed"], N_Ed=sc["N_Ed"])
    d = result.details
    status = result.status.value.upper()

    print(f"\n{sc['name']}:")
    print(f"  M_Ed = {sc['M_Ed']:5.0f} kN\u00b7m, N_Ed = {sc['N_Ed']:5.0f} kN")
    if d.get('is_cracked', True):
        print(f"  Section CRACKED")
        print(f"  \u03c3_s = {d.get('sigma_s', 0):.1f} MPa, s_r,max = {d.get('s_r_max', 0):.1f} mm")
        print(f"  w_k = {d.get('w_k', 0):.3f} mm \u2264 {check.w_k_limit} mm")
    else:
        print(f"  Section UNCRACKED (M_Ed < M_cr)")
        print(f"  w_k = 0 mm")
    print(f"  Utilization: {result.utilization:.1%} [{status}]")

Cracking moment M_cr (N=0): 72.0 kN·m

Cracking Check Results

Low SLS moment:
  M_Ed =    40 kN·m, N_Ed =     0 kN
  Section UNCRACKED (M_Ed < M_cr)
  w_k = 0 mm
  Utilization: 0.0% [PASS]



Service moment:
  M_Ed =    80 kN·m, N_Ed =     0 kN
  Section CRACKED
  σ_s = 164.7 MPa, s_r,max = 711.3 mm
  w_k = 0.352 mm ≤ 0.3 mm
  Utilization: 117.2% [FAIL]

With axial compression:
  M_Ed =   100 kN·m, N_Ed =   200 kN
  Section CRACKED
  σ_s = 136.1 MPa, s_r,max = 660.2 mm
  w_k = 0.270 mm ≤ 0.3 mm
  Utilization: 89.9% [PASS]

High moment + axial:
  M_Ed =   120 kN·m, N_Ed =   200 kN
  Section CRACKED
  σ_s = 176.8 MPa, s_r,max = 670.0 mm
  w_k = 0.355 mm ≤ 0.3 mm
  Utilization: 118.4% [FAIL]


### 5.2 Cracking Moment with Axial Load

`find_cracking_moment(N_Ed)` now supports axial load. Compression increases
M_cr (delays cracking), tension decreases it.

In [44]:
axial_loads = [-200, -100, 0, 200, 500, 1000, 1500]

print("Cracking Moment vs Axial Load")
print("=" * 40)
print(f"{'N_Ed (kN)':>12} | {'M_cr (kN\u00b7m)':>14}")
print("-" * 40)

for N in axial_loads:
    M_cr_N = check.cracking.find_cracking_moment(N_Ed=N)
    print(f"{N:>12.0f} | {M_cr_N:>14.1f}")

Cracking Moment vs Axial Load
   N_Ed (kN) |    M_cr (kN·m)
----------------------------------------
        -200 |           56.8
        -100 |           64.4
           0 |           72.0
         200 |           87.2
         500 |          110.1
        1000 |          148.1
        1500 |          186.1


### 5.3 Force-Cracked Mode

For piles and columns that always have reinforcement, the cracking check can
be forced into cracked analysis regardless of M_cr. This is useful when the
section is expected to crack during its service life.

In [45]:
# Compare auto vs force_cracked for a low-moment case
M_sls = 40  # Below M_cr

result_auto = check.perform_cracking_check(M_Ed=M_sls, N_Ed=0)
result_forced = check.perform_cracking_check(M_Ed=M_sls, N_Ed=0, force_cracked=True)

print(f"force_cracked Comparison (M_Ed = {M_sls} kN\u00b7m, M_cr = {check.cracking.find_cracking_moment():.1f} kN\u00b7m)")
print("=" * 55)
print(f"{'Parameter':<25} | {'Auto':>12} | {'Forced':>12}")
print("-" * 55)
print(f"{'Is cracked':<25} | {str(result_auto.details.get('is_cracked', False)):>12} | {str(result_forced.details.get('is_cracked', True)):>12}")
print(f"{'w_k (mm)':<25} | {result_auto.details.get('w_k', 0):>12.3f} | {result_forced.details.get('w_k', 0):>12.3f}")
print(f"{'Utilization':<25} |  {result_auto.utilization:>11.1%} |  {result_forced.utilization:>11.1%}")

force_cracked Comparison (M_Ed = 40 kN·m, M_cr = 72.0 kN·m)
Parameter                 |         Auto |       Forced
-------------------------------------------------------
Is cracked                |        False |         True
w_k (mm)                  |        0.000 |        0.176
Utilization               |         0.0% |        58.6%


---
## 6. k_f Factor for Cast-in-Place Piles (EC2 §2.4.2.5)

EC2 §2.4.2.5(2) requires γ_c to be multiplied by k_f for cast-in-place piles
without permanent casing. The NDP default is k_f = 1.1.

When `apply_k_f=True`:
- ULS concrete (γ_c = 1.5, k_f = 1.1: = 1.65) gives reduced f_cd, f_ctd
- SLS cracking uses original concrete (no k_f)

In [46]:
from materials.reinforced_concrete.ndp import get_ndp

k_f = get_ndp("k_f")

# Create pile check with k_f applied
check_pile = CircularSectionCheck(
    section=section,
    concrete=concrete,
    diameter=pile_dia,
    cover=cover,
    shear_reinforcement=links,
    apply_k_f=True,
)

print(f"k_f factor (NDP): {k_f}")
print(f"Standard \u03b3_c:     {concrete.gamma_c}")
print(f"Pile \u03b3_c:         {concrete.gamma_c * k_f:.2f}")
print(f"Standard f_cd:    {concrete.f_cd:.2f} MPa")
print(f"Pile f_cd:        {concrete.f_ck / (concrete.gamma_c * k_f):.2f} MPa")
print()

# Compare bending capacity
M_test, N_test = 150, 500
result_std = check.perform_bending_check(M_Ed=M_test, N_Ed=N_test)
result_pile = check_pile.perform_bending_check(M_Ed=M_test, N_Ed=N_test)

print(f"Bending Comparison (M_Ed={M_test} kN\u00b7m, N_Ed={N_test} kN)")
print("=" * 50)
print(f"{'Parameter':<25} | {'Standard':>10} | {'With k_f':>10}")
print("-" * 50)
print(f"{'M_Rd (kN\u00b7m)':<25} | {result_std.details['M_Rd']:>10.1f} | {result_pile.details['M_Rd']:>10.1f}")
print(f"{'Utilization':<25} | {result_std.utilization:>9.1%} | {result_pile.utilization:>9.1%}")

# Compare shear capacity
print()
shear_lc = ShearLoadCase(V_Ed=200, M_Ed=150, N_Ed=500)
result_s_std = check.perform_shear_check(load_case=shear_lc)
result_s_pile = check_pile.perform_shear_check(load_case=shear_lc)

print(f"Shear Comparison (V_Ed={shear_lc.V_Ed} kN)")
print("=" * 50)
print(f"{'Parameter':<25} | {'Standard':>10} | {'With k_f':>10}")
print("-" * 50)
print(f"{'V_Rd (kN)':<25} | {result_s_std.capacity:>10.1f} | {result_s_pile.capacity:>10.1f}")
print(f"{'Utilization':<25} | {result_s_std.utilization:>9.1%} | {result_s_pile.utilization:>9.1%}")

# Cracking should be identical (no k_f at SLS)
print()
result_c_std = check.perform_cracking_check(M_Ed=80, N_Ed=0)
result_c_pile = check_pile.perform_cracking_check(M_Ed=80, N_Ed=0)
print(f"Cracking: standard w_k = {result_c_std.details.get('w_k', 0):.3f} mm, "
      f"pile w_k = {result_c_pile.details.get('w_k', 0):.3f} mm (should be identical)")

k_f factor (NDP): 1.1
Standard γ_c:     1.5
Pile γ_c:         1.65
Standard f_cd:    20.00 MPa
Pile f_cd:        18.18 MPa

Bending Comparison (M_Ed=150 kN·m, N_Ed=500 kN)
Parameter                 |   Standard |   With k_f
--------------------------------------------------
M_Rd (kN·m)               |      475.5 |      452.0
Utilization               |     31.5% |     33.2%

Shear Comparison (V_Ed=200.0 kN)
Parameter                 |   Standard |   With k_f
--------------------------------------------------
V_Rd (kN)                 |      467.7 |      426.1
Utilization               |     42.8% |     46.9%

Cracking: standard w_k = 0.352 mm, pile w_k = 0.352 mm (should be identical)


---
## 7. Spiral Links

When `is_spiral=True`, `ShearRebar.link_spacing` is treated as the spiral pitch
for the λ₂ calculation (Eq.8). λ₂ reduces V_Rd,s to account for the helix angle:

$$\lambda_2 = \left(\left(\frac{p}{2\pi r_{sv}}\right)^2 + 1\right)^{-0.5}$$

In [47]:
# Create check with spiral links (same spacing = pitch)
spiral_links = ShearRebar(diameter=12, link_spacing=150, n_legs=2, grade="B500B")

check_spiral = CircularSectionCheck(
    section=section,
    concrete=concrete,
    diameter=pile_dia,
    cover=cover,
    shear_reinforcement=spiral_links,
    is_spiral=True,
)

check_closed = CircularSectionCheck(
    section=section,
    concrete=concrete,
    diameter=pile_dia,
    cover=cover,
    shear_reinforcement=spiral_links,
    is_spiral=False,
)

print(f"Spiral pitch (= spacing): {spiral_links.link_spacing} mm")
print(f"r_sv: {check_spiral.r_sv:.1f} mm")
print(f"\u03bb\u2082 (closed links): {check_closed.calculate_lambda_2():.4f}")
print(f"\u03bb\u2082 (spiral):       {check_spiral.calculate_lambda_2():.4f}")
print()

# Compare shear capacities
test_lc = ShearLoadCase(V_Ed=250, M_Ed=150, N_Ed=500)
result_closed = check_closed.perform_shear_check(load_case=test_lc)
result_spiral = check_spiral.perform_shear_check(load_case=test_lc)

print("Closed Links vs Spiral Links")
print("=" * 52)
print(f"{'Parameter':<25} | {'Closed':>10} | {'Spiral':>10}")
print("-" * 52)
print(f"{'\u03bb\u2082':<25} | {result_closed.details['lambda_2']:>10.4f} | {result_spiral.details['lambda_2']:>10.4f}")
print(f"{'V_Rd,s (kN)':<25} | {result_closed.details['V_Rd_s']:>10.1f} | {result_spiral.details['V_Rd_s']:>10.1f}")
print(f"{'V_Rd,max (kN)':<25} | {result_closed.details['V_Rd_max']:>10.1f} | {result_spiral.details['V_Rd_max']:>10.1f}")
print(f"{'Utilization':<25} |  {result_closed.utilization:>9.1%} |  {result_spiral.utilization:>9.1%}")

Spiral pitch (= spacing): 150.0 mm
r_sv: 244.0 mm
λ₂ (closed links): 1.0000
λ₂ (spiral):       0.9952

Closed Links vs Spiral Links
Parameter                 |     Closed |     Spiral
----------------------------------------------------
λ₂                        |     1.0000 |     0.9952
V_Rd,s (kN)               |      494.0 |      491.7
V_Rd,max (kN)             |      467.7 |      467.7
Utilization               |      53.5% |      53.5%


### 7.1 λ₂ Sensitivity to Spiral Pitch

λ₂ varies with the ratio of pitch to circumference. Tighter spirals (smaller p)
give higher λ₂ (closer to 1.0).

In [48]:
from math import pi

r_sv = check.r_sv
pitches = [50, 75, 100, 125, 150, 200, 250, 300]

print(f"\u03bb\u2082 vs Spiral Pitch (r_sv = {r_sv:.0f} mm)")
print("=" * 40)
print(f"{'Pitch (mm)':>12} | {'\u03bb\u2082':>10} | {'Reduction':>10}")
print("-" * 40)

for p in pitches:
    lambda_2 = ((p / (2 * pi * r_sv)) ** 2 + 1) ** (-0.5)
    print(f"{p:>12.0f} | {lambda_2:>10.4f} | {(1 - lambda_2) * 100:>9.1f}%")

λ₂ vs Spiral Pitch (r_sv = 242 mm)
  Pitch (mm) |         λ₂ |  Reduction
----------------------------------------
          50 |     0.9995 |       0.1%
          75 |     0.9988 |       0.1%
         100 |     0.9978 |       0.2%
         125 |     0.9966 |       0.3%
         150 |     0.9952 |       0.5%
         200 |     0.9915 |       0.9%
         250 |     0.9868 |       1.3%
         300 |     0.9811 |       1.9%


---
## 8. V_Rd,c for Uncracked Sections (Eq.17)

The uncracked shear capacity based on principal tensile stress is:

$$V_{Rd,c} = \frac{3\pi r^2}{4} \sqrt{f_{ctd}^2 + \sigma_{cp} \cdot f_{ctd}}$$

Axial compression (\u03c3_cp > 0) increases V_Rd,c by delaying the onset of
diagonal cracking.

In [49]:
sigma_cps = np.linspace(0, 10, 20)

print(f"V_Rd,c (uncracked) vs \u03c3_cp")
print(f"D = {pile_dia} mm, f_ctd = {check._f_ctd_design:.2f} MPa")
print("=" * 40)
print(f"{'\u03c3_cp (MPa)':>12} | {'V_Rd,c (kN)':>12}")
print("-" * 30)

for sigma in sigma_cps[::3]:  # Show every 3rd value
    V_Rd_c = check.calculate_V_Rd_c_uncracked(float(sigma))
    print(f"{sigma:>12.2f} | {V_Rd_c:>12.1f}")

V_Rd,c (uncracked) vs σ_cp
D = 600 mm, f_ctd = 1.35 MPa
  σ_cp (MPa) |  V_Rd,c (kN)
------------------------------
        0.00 |        286.6
        1.58 |        422.1
        3.16 |        523.6
        4.74 |        608.3
        6.32 |        682.7
        7.89 |        749.7
        9.47 |        811.2


---
## 9. Equivalent Web Width for Circular Sections

The strut crushing check (V_Rd,max) uses an equivalent rectangular web width:

- **b_wc** = width at compression centroid depth (Eq.10)
- **b_wt** = width inside shear reinforcement at tension centroid (Eq.12)
- **b_w** = min(b_wc, b_wt)

In [50]:
# Show how b_w varies with effective depth / lever arm
ds = np.linspace(350, 550, 10)

print(f"Equivalent Web Width vs Effective Depth")
print(f"D = {pile_dia} mm, r_sv = {check.r_sv:.0f} mm")
print("=" * 60)
print(f"{'d (mm)':>8} | {'z (mm)':>8} | {'b_wc (mm)':>10} | {'b_wt (mm)':>10} | {'b_w (mm)':>10}")
print("-" * 60)

for d in ds:
    z = 0.8 * d  # Approximate lever arm for display
    b_w, b_wc, b_wt = check.calculate_equivalent_web_width(float(d), float(z))
    print(f"{d:>8.0f} | {z:>8.0f} | {b_wc:>10.1f} | {b_wt:>10.1f} | {b_w:>10.1f}")

Equivalent Web Width vs Effective Depth
D = 600 mm, r_sv = 242 mm
  d (mm) |   z (mm) |  b_wc (mm) |  b_wt (mm) |   b_w (mm)
------------------------------------------------------------
     350 |      280 |      385.2 |      473.6 |      385.2
     372 |      298 |      395.6 |      461.9 |      395.6
     394 |      316 |      405.5 |      445.6 |      405.5
     417 |      333 |      415.0 |      424.0 |      415.0
     439 |      351 |      424.1 |      396.4 |      396.4
     461 |      369 |      432.8 |      361.1 |      361.1
     483 |      387 |      441.2 |      315.9 |      315.9
     506 |      404 |      449.2 |      255.4 |      255.4
     528 |      422 |      456.9 |      163.5 |      163.5
     550 |      440 |      464.3 |        0.0 |      464.3


---
## 10. Full Combined Check

Run all three checks for a typical pile design scenario.

In [51]:
# ULS loads
M_Ed_uls = 180   # kN·m
V_Ed_uls = 250   # kN
N_Ed_uls = 600   # kN

# SLS loads
M_Ed_sls = 120   # kN·m
N_Ed_sls = 400   # kN

print(f"FULL DESIGN CHECK - {pile_dia}mm Circular Pile")
print("=" * 60)
print(f"ULS: M_Ed = {M_Ed_uls} kN\u00b7m, V_Ed = {V_Ed_uls} kN, N_Ed = {N_Ed_uls} kN")
print(f"SLS: M_Ed = {M_Ed_sls} kN\u00b7m, N_Ed = {N_Ed_sls} kN")

# 1. Bending (with tension shift)
M_cap = check.bending.get_moment_capacity(N_Ed=N_Ed_uls)[0]
bending_result = check.perform_bending_check(
    M_Ed=M_Ed_uls, N_Ed=N_Ed_uls,
    V_Ed=V_Ed_uls, M_cap=M_cap,
)
print(f"\n1. BENDING (EC2 \u00a76.1)")
print(f"   M_Rd = {bending_result.details['M_Rd']:.1f} kN\u00b7m")
print(f"   Utilization: {bending_result.utilization:.1%} [{bending_result.status.value.upper()}]")

# 2. Shear (force cracked for pile)
shear_result = check.perform_shear_check(
    load_case=ShearLoadCase(V_Ed=V_Ed_uls, M_Ed=M_Ed_uls, N_Ed=N_Ed_uls),
)
sd = shear_result.details
print(f"\n2. SHEAR (Orr 2012)")
if 'V_Rd_s' in sd and sd['V_Rd_s'] is not None:
    print(f"   \u03bb\u2081 = {sd['lambda_1']:.3f}, b_w = {sd['b_w']:.0f} mm")
    print(f"   V_Rd,s = {sd['V_Rd_s']:.1f} kN, V_Rd,max = {sd['V_Rd_max']:.1f} kN")
print(f"   V_Rd = {shear_result.capacity:.1f} kN")
print(f"   Utilization: {shear_result.utilization:.1%} [{shear_result.status.value.upper()}]")

# 3. Cracking
cracking_result = check.perform_cracking_check(
    M_Ed=M_Ed_sls, N_Ed=N_Ed_sls,
)
cd = cracking_result.details
print(f"\n3. CRACKING (EC2 \u00a77.3)")
if cd.get('is_cracked', True):
    print(f"   w_k = {cd.get('w_k', 0):.3f} mm \u2264 {check.w_k_limit} mm")
else:
    print(f"   Section uncracked (w_k = 0)")
print(f"   Utilization: {cracking_result.utilization:.1%} [{cracking_result.status.value.upper()}]")

# Summary
from materials.reinforced_concrete.code_checks.base_check import CheckStatus
all_pass = all(
    r.status in (CheckStatus.PASS, CheckStatus.WARNING)
    for r in [bending_result, shear_result, cracking_result]
)
print(f"\n{'=' * 60}")
print(f"OVERALL: {'ALL CHECKS PASS' if all_pass else 'SOME CHECKS FAIL'}")

FULL DESIGN CHECK - 600mm Circular Pile
ULS: M_Ed = 180 kN·m, V_Ed = 250 kN, N_Ed = 600 kN
SLS: M_Ed = 120 kN·m, N_Ed = 400 kN

1. BENDING (EC2 §6.1)
   M_Rd = 407.6 kN·m
   Utilization: 72.8% [PASS]

2. SHEAR (Orr 2012)
   λ₁ = 0.850, b_w = 365 mm
   V_Rd,s = 655.7 kN, V_Rd,max = 469.0 kN
   V_Rd = 469.0 kN
   Utilization: 53.3% [PASS]

3. CRACKING (EC2 §7.3)
   w_k = 0.211 mm ≤ 0.3 mm
   Utilization: 70.4% [PASS]

OVERALL: ALL CHECKS PASS


---
## 11. Accessing Sub-Checks Directly

The internal `BendingCheck` and `CrackingCheck` are exposed via properties
for advanced operations like plotting, capacity queries, or detailed calculations.

In [52]:
# Access internal BendingCheck for moment capacity at various axial loads
axial_loads = [0, 200, 500, 1000, 1500, -200]

print("Moment Capacity (via check.bending)")
print("=" * 50)
print(f"{'N_Ed (kN)':>12} | {'M_Rd+ (kN\u00b7m)':>14} | {'M_Rd- (kN\u00b7m)':>14}")
print("-" * 50)

for N in axial_loads:
    M_pos, M_neg = check.bending.get_moment_capacity(N_Ed=N)
    M_pos_str = f"{M_pos:.1f}" if M_pos is not None else "N/A"
    M_neg_str = f"{M_neg:.1f}" if M_neg is not None else "N/A"
    print(f"{N:>12.0f} | {M_pos_str:>14} | {M_neg_str:>14}")

Moment Capacity (via check.bending)
   N_Ed (kN) |   M_Rd+ (kN·m) |   M_Rd- (kN·m)
--------------------------------------------------
           0 |          290.5 |         -290.5
         200 |          323.8 |         -323.8
         500 |          370.4 |         -370.4
        1000 |          425.7 |         -425.7
        1500 |          469.8 |         -469.8
        -200 |          256.3 |         -256.3


In [53]:
# Stress-strain profile from the bending sub-check
check.bending.plot_stress_strain(
    M_Ed=180,
    N_Ed=600,
    title="Stress-Strain Distribution (M=180 kN\u00b7m, N=600 kN)",
    section_render="filled",
    height=800,
    show=False,
)

In [ ]:
# Detailed cracking result via check.cracking
detailed = check.cracking.calculate_detailed(M_Ed=120, N_Ed=400)

print("Detailed Cracking (via check.cracking)")
print("=" * 45)
print(f"Is cracked:        {detailed.is_cracked}")
if detailed.is_cracked:
    print(f"N.A. depth x:  {detailed.x:.1f} mm")
    print(f"\u03c3_s:      {detailed.sigma_s:.1f} MPa")
    print(f"\u03c1_p,eff:  {detailed.rho_p_eff:.4f}")
    print(f"s_r,max:       {detailed.s_r_max:.1f} mm")
    print(f"w_k:           {detailed.w_k:.3f} mm")
else:
    print(f"Section uncracked")

Detailed Cracking (via check.cracking)
Is cracked:    True
N.A. depth x:  272.6 mm
σ_s:           114.3 MPa
ρ_p,eff:       0.0096
s_r,max:       616.0 mm
w_k:           0.211 mm


---
## 12. ShearRebar.angle Warning

The `angle` parameter on `ShearRebar` is ineffective for circular sections
(only 90\u00b0 vertical links/spirals are supported by the λ₁/λ₂ formulation).
`CircularSectionCheck` emits a warning if a non-90\u00b0 angle is provided.

In [56]:
# Demonstrate the angle warning
inclined_links = ShearRebar(diameter=12, link_spacing=200, n_legs=2, angle=60)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    check_angled = CircularSectionCheck(
        section=section,
        concrete=concrete,
        diameter=pile_dia,
        cover=cover,
        shear_reinforcement=inclined_links,
    )
    if w:
        print(f"Warning raised: {w[0].message}")
    else:
        print("No warnings raised")

Warning raised: ShearRebar.angle=60.0° is ignored for circular sections — links must be 90° (vertical).Spirals use spacing for pitch, angle makes no difference. The λ1/λ2 efficiency factors account for circular geometry.


---
## Summary

`CircularSectionCheck` provides an EC2-compliant design workflow for circular
RC sections following Orr (2012). Key features demonstrated:

| Feature | Section | Method / Parameter |
|---------|---------|--------------------|
| Bending with M-N diagram | \u00a73 | `perform_bending_check()` |
| Tension shift (circular b_w) | \u00a73.3 | Auto cot(\u03b8) from circular geometry |
| Shear with \u03bb\u2081/\u03bb\u2082 | \u00a74 | `perform_shear_check()` |
| \u03bb\u2081 numerical vs simplified | \u00a74.4 | `use_simplified_lambda_1=True` |
| Cracking (pass-through) | \u00a75 | `perform_cracking_check()` |
| M_cr with axial load | \u00a75.2 | `cracking.find_cracking_moment(N_Ed=...)` |
| Force cracked mode (cracking) | \u00a75.3 | `perform_cracking_check(..., force_cracked=True)` |
| k_f for piles | \u00a76 | `apply_k_f=True` |
| Spiral links | \u00a77 | `is_spiral=True` |
| V_Rd,c uncracked | \u00a78 | `calculate_V_Rd_c_uncracked()` |
| Equivalent web width | \u00a79 | `calculate_equivalent_web_width()` |
| Sub-check access | \u00a711 | `check.bending`, `check.cracking` |